# Plant Leaf 4x SR — 3-Phase cGAN (v6, 6h, 2×T4)

## Architecture
- **Generator**: 20 Residual Blocks + bicubic skip connection
- **Discriminator**: Conditional PatchGAN (6-ch input: concat bicubic(LR) + HR)
- **Loss**: LSGAN
- **Multi-GPU**: DataParallel across both T4s

## Per-Phase Batch Sizes
| Phase | Batch | Batches/ep | s/batch | s/ep |
|-------|-------|-----------|---------|------|
| 1 (L1 only) | **128** | 13 | ~5.0s | **~65s** |
| 2 (L1+VGG+Struct) | **64** | 26 | ~3.0s | **~78s** |
| 3 (G+D) | **32** | 52 | ~2.1s | **~109s** |

## Timing Budget — confirmed from live run (5.0s/batch at BS=128)
- Phase 1: 120 × ~65s = **~130 min**
- Phase 2:  75 × ~78s =  **~98 min**
- Phase 3:  55 × ~109s = **~100 min**
- MAE evals (10×~1.5 min): **~15 min**
- **Total ≈ 343 min ≈ 5h 43min** ✓

## Inference: 7-fold TTA

In [1]:
import os, random, shutil
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

In [2]:
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
NUM_GPUS = torch.cuda.device_count()
print(f"Device: {DEVICE}  |  GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  "
          f"VRAM={torch.cuda.get_device_properties(i).total_memory//1024**2} MB")

TRAIN_HR_DIR     = "/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_High_Resolution/"
TRAIN_LR_DIR     = "/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution/"
TEST_LR_DIR      = "/kaggle/input/competitions/plant-leaves-super-resolution-challenge/test_Low_Resolution/"
VGG_WEIGHTS_PATH = "/kaggle/input/competitions/plant-leaves-super-resolution-challenge/vgg19_weights.pth"

NUM_BLOCKS = 20

# ── Per-phase batch sizes (calibrated from live run: 5.0s/batch at BS=128) ────
# Phase 1 (pure L1, no VGG):  BS=128 → 13 batches/ep × 5.0s = ~65s/ep
# Phase 2 (L1+VGG+Struct):    BS=64  → 26 batches/ep × 3.0s = ~78s/ep
# Phase 3 (G+D+VGG):          BS=32  → 52 batches/ep × 2.1s = ~109s/ep
BS_P1 = 128    # 64 samples/GPU  — 2.5 GB VRAM/GPU (measured)
BS_P2 = 64     # 32 samples/GPU  — est. ~5-7 GB VRAM/GPU with VGG
BS_P3 = 32     # 16 samples/GPU  — est. ~6-8 GB VRAM/GPU with G+D+VGG

# ── Epochs (calibrated to fit 6h: 120×65 + 75×78 + 55×109 + evals = ~343 min) ─
PHASE1_EPOCHS = 120
PHASE2_EPOCHS = 75
PHASE3_EPOCHS = 55

# ── Learning rates ────────────────────────────────────────────────────────────
P1_LR   = 3e-4    # Phase 1: modest sqrt-scale of 2e-4 for BS=128 vs BS=16
P2_LR   = 1e-4    # Phase 2: perceptual refinement
P3_LR_G = 5e-5    # Phase 3: GAN generator
P3_LR_D = 1e-5    # Phase 3: GAN discriminator

print("=" * 65)
print("3-PHASE cGAN  |  6-HOUR BUDGET  |  v6  |  2xT4")
print(f"Blocks={NUM_BLOCKS}")
print(f"P1={PHASE1_EPOCHS}ep  BS={BS_P1} ({BS_P1//max(NUM_GPUS,1)}/GPU)  ~65s/ep  LR={P1_LR:.0e}")
print(f"P2={PHASE2_EPOCHS}ep  BS={BS_P2} ({BS_P2//max(NUM_GPUS,1)}/GPU)  ~78s/ep  LR={P2_LR:.0e}")
print(f"P3={PHASE3_EPOCHS}ep  BS={BS_P3} ({BS_P3//max(NUM_GPUS,1)}/GPU)  ~109s/ep LR_G={P3_LR_G:.0e}")
print("=" * 65)

Device: cuda  |  GPUs: 2
  GPU 0: Tesla T4  VRAM=14912 MB
  GPU 1: Tesla T4  VRAM=14912 MB
3-PHASE cGAN  |  6-HOUR BUDGET  |  v6  |  2xT4
Blocks=20
P1=120ep  BS=128 (64/GPU)  ~65s/ep  LR=3e-04
P2=75ep  BS=64 (32/GPU)  ~78s/ep  LR=1e-04
P3=55ep  BS=32 (16/GPU)  ~109s/ep LR_G=5e-05


In [3]:
class SRDataset(Dataset):
    """Paired hflip augmentation — LR and HR flipped together."""
    def __init__(self, lr_dir, hr_dir, augment=True):
        self.lr_dir = lr_dir; self.hr_dir = hr_dir
        self.files  = sorted(os.listdir(lr_dir)); self.augment = augment
        self.tf = transforms.Compose([transforms.ToTensor(),
                                      transforms.Normalize([0.5]*3, [0.5]*3)])
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        name  = self.files[idx]
        lr_im = Image.open(os.path.join(self.lr_dir, name)).convert("RGB")
        hr_im = Image.open(os.path.join(self.hr_dir, name)).convert("RGB")
        if self.augment and random.random() > 0.5:
            lr_im = transforms.functional.hflip(lr_im)
            hr_im = transforms.functional.hflip(hr_im)
        return self.tf(lr_im), self.tf(hr_im)


class SRTestDataset(Dataset):
    def __init__(self, lr_dir):
        self.lr_dir = lr_dir; self.files = sorted(os.listdir(lr_dir))
        self.tf = transforms.Compose([transforms.ToTensor(),
                                      transforms.Normalize([0.5]*3, [0.5]*3)])
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        name = self.files[idx]
        return self.tf(Image.open(os.path.join(self.lr_dir, name)).convert("RGB")), name


def make_loader(batch_size):
    """Create a training DataLoader for the given batch size."""
    return DataLoader(
        SRDataset(TRAIN_LR_DIR, TRAIN_HR_DIR, augment=True),
        batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True
    )

In [4]:
# ── Generator (SRGAN-style: 20 Residual Blocks + bicubic skip) ────────────────
class ResidualBlock(nn.Module):
    def __init__(self, ch=64):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(ch,ch,3,1,1), nn.PReLU(), nn.Conv2d(ch,ch,3,1,1))
    def forward(self, x): return x + 0.1 * self.block(x)

class Generator(nn.Module):
    def __init__(self, num_blocks=NUM_BLOCKS):
        super().__init__()
        self.initial = nn.Sequential(nn.Conv2d(3, 64, 9, 1, 4), nn.PReLU())
        self.res   = nn.Sequential(*[ResidualBlock(64) for _ in range(num_blocks)])
        self.mid   = nn.Conv2d(64, 64, 3, 1, 1)
        self.up    = nn.Sequential(
            nn.Conv2d(64, 256, 3, 1, 1), nn.PixelShuffle(2), nn.PReLU(),
            nn.Conv2d(64, 256, 3, 1, 1), nn.PixelShuffle(2), nn.PReLU())
        self.final = nn.Conv2d(64, 3, 9, 1, 4)
    def forward(self, x):
        feat  = self.initial(x)
        feat  = feat + self.mid(self.res(feat))
        out   = self.final(self.up(feat))
        lr_up = F.interpolate(x, scale_factor=4, mode='bicubic', align_corners=False)
        return torch.clamp(out + lr_up, -1, 1)


# ── Conditional PatchGAN Discriminator ────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=2, bn=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 3, stride, 1)]
        if bn: layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class Discriminator(nn.Module):
    """Conditional PatchGAN: concat(bicubic(lr), hr_or_fake) -> 6ch -> patch scores."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(6,  64,  stride=2, bn=False),
            ConvBlock(64, 128, stride=2),
            ConvBlock(128,256, stride=2),
            ConvBlock(256,512, stride=1),
            nn.Conv2d(512, 1, 3, 1, 1),
        )
    def forward(self, lr_img, hr_img):
        lr_up = F.interpolate(lr_img, scale_factor=4, mode='bicubic', align_corners=False)
        return self.net(torch.cat([lr_up, hr_img], dim=1))

def weight_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

def wrap_dp(module):
    return nn.DataParallel(module) if NUM_GPUS > 1 else module

In [5]:
class VGGLoss(nn.Module):
    def __init__(self, path):
        super().__init__()
        vgg = models.vgg19(weights=None)
        vgg.load_state_dict(torch.load(path, map_location="cpu"))
        self.features = vgg.features[:36].eval()
        for p in self.features.parameters(): p.requires_grad = False
        self.l1 = nn.L1Loss()
    def preprocess(self, x):
        x = (x + 1) / 2
        mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(x.device)
        std  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(x.device)
        return (x - mean) / std
    def forward(self, fake, real):
        return self.l1(self.features(self.preprocess(fake)),
                       self.features(self.preprocess(real)))


def lsgan_d_loss(d_real, d_fake):
    return (0.5 * F.mse_loss(d_real, torch.ones_like(d_real)) +
            0.5 * F.mse_loss(d_fake, torch.zeros_like(d_fake)))

def lsgan_g_loss(d_fake):
    return F.mse_loss(d_fake, torch.ones_like(d_fake))


@torch.no_grad()
def full_eval_mae(model, lr_dir, hr_dir):
    """MAE in [0,255]. Unwraps DataParallel for single-image inference."""
    eval_model = model.module if hasattr(model, "module") else model
    eval_model.eval()
    tf_lr = transforms.Compose([transforms.ToTensor(),
                                 transforms.Normalize([0.5]*3, [0.5]*3)])
    tf_hr = transforms.ToTensor()
    total = 0.0
    files = sorted(os.listdir(lr_dir))
    for fname in tqdm(files, desc="MAE eval", leave=False):
        lr   = tf_lr(Image.open(os.path.join(lr_dir, fname)).convert("RGB")).unsqueeze(0).to(DEVICE)
        hr   = tf_hr(Image.open(os.path.join(hr_dir, fname)).convert("RGB")).to(DEVICE)
        pred = ((eval_model(lr)+1)/2).clamp(0,1).squeeze(0)
        total += (pred - hr).abs().mean().item() * 255
    avg = total / len(files)
    print(f"  MAE: {avg:.4f}")
    model.train()
    return avg


def save_G(G, path):
    state = G.module.state_dict() if hasattr(G, "module") else G.state_dict()
    torch.save(state, path)

def load_G(G, path):
    state = torch.load(path, map_location=DEVICE)
    if hasattr(G, "module"):
        G.module.load_state_dict(state)
    else:
        G.load_state_dict(state)

In [6]:
# ============================================================
# PHASE 1 — Pure L1 Pixel Warm-Up
# BS=128 → 13 batches/ep × ~5.0s/batch = ~65s/ep (measured)
# 120 ep × 65s = ~130 min
# ============================================================
print("=" * 65)
print(f"PHASE 1: Pure L1 Pixel Warm-Up  ({PHASE1_EPOCHS} epochs, BS={BS_P1})")
print("=" * 65)

G      = wrap_dp(Generator(num_blocks=NUM_BLOCKS).to(DEVICE))
loader = make_loader(BS_P1)
opt_G  = optim.Adam(G.parameters(), lr=P1_LR, weight_decay=1e-6)
sch_G  = optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=PHASE1_EPOCHS, eta_min=1e-6)

best_mae_p1  = float("inf")
best_loss_p1 = float("inf")

for epoch in range(1, PHASE1_EPOCHS + 1):
    G.train(); losses = []

    for lr_b, hr_b in tqdm(loader, desc=f"P1 Ep {epoch}/{PHASE1_EPOCHS}", leave=False):
        lr_b, hr_b = lr_b.to(DEVICE), hr_b.to(DEVICE)
        pred = G(lr_b)
        loss = F.l1_loss(pred, hr_b)
        opt_G.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        opt_G.step()
        losses.append(loss.item())

    sch_G.step()
    avg = sum(losses) / len(losses)
    print(f"P1 Ep {epoch:03d}/{PHASE1_EPOCHS} | L1={avg:.5f} | LR={sch_G.get_last_lr()[0]:.6f}")

    if avg < best_loss_p1:
        best_loss_p1 = avg; save_G(G, "p1_best.pth")

    # MAE eval at ep40, ep80, ep120
    if epoch % 40 == 0 or epoch == PHASE1_EPOCHS:
        mae = full_eval_mae(G, TRAIN_LR_DIR, TRAIN_HR_DIR)
        if mae < best_mae_p1:
            best_mae_p1 = mae; save_G(G, "p1_best_mae.pth")
            print(f"  --> p1_best_mae.pth  (MAE={best_mae_p1:.4f})")

if not os.path.exists("p1_best_mae.pth"):
    save_G(G, "p1_best_mae.pth")
    print("  [fallback] saved final epoch as p1_best_mae.pth")

print(f"\nPhase 1 done | best MAE={best_mae_p1:.4f}")

PHASE 1: Pure L1 Pixel Warm-Up  (120 epochs, BS=128)


P1 Ep 001/120 | L1=0.14535 | LR=0.000300


P1 Ep 002/120 | L1=0.14303 | LR=0.000300


P1 Ep 003/120 | L1=0.14164 | LR=0.000300


P1 Ep 004/120 | L1=0.14076 | LR=0.000299


...

P1 Ep 117/120 | L1=0.13459 | LR=0.000001


P1 Ep 118/120 | L1=0.13450 | LR=0.000001


P1 Ep 119/120 | L1=0.13456 | LR=0.000001


P1 Ep 120/120 | L1=0.13457 | LR=0.000001


  MAE: 17.1566
  --> p1_best_mae.pth  (MAE=17.1566)

Phase 1 done | best MAE=17.1566


In [7]:
# ============================================================
# PHASE 2 — Perceptual Refinement
# BS=64 → 26 batches/ep × ~3.0s/batch = ~78s/ep
# 75 ep × 78s = ~98 min
# ============================================================
print("=" * 65)
print(f"PHASE 2: Perceptual Refinement  ({PHASE2_EPOCHS} epochs, BS={BS_P2})")
print("=" * 65)

for ckpt in ["p1_best_mae.pth", "p1_best.pth"]:
    if os.path.exists(ckpt):
        load_G(G, ckpt); print(f"Loaded P1 weights: {ckpt}"); break

VGG_loss = VGGLoss(VGG_WEIGHTS_PATH).to(DEVICE)
loader   = make_loader(BS_P2)
opt_G2   = optim.Adam(G.parameters(), lr=P2_LR, weight_decay=1e-6)
sch_G2   = optim.lr_scheduler.CosineAnnealingLR(opt_G2, T_max=PHASE2_EPOCHS, eta_min=1e-6)

best_mae_p2  = best_mae_p1
best_loss_p2 = float("inf")

for epoch in range(1, PHASE2_EPOCHS + 1):
    G.train(); losses = []

    for lr_b, hr_b in tqdm(loader, desc=f"P2 Ep {epoch}/{PHASE2_EPOCHS}", leave=False):
        lr_b, hr_b = lr_b.to(DEVICE), hr_b.to(DEVICE)
        pred = G(lr_b)

        l_pixel  = F.l1_loss(pred, hr_b)
        l_vgg    = VGG_loss(pred, hr_b)
        down     = F.interpolate(pred, scale_factor=0.25, mode="bicubic", align_corners=False)
        l_struct = F.l1_loss(down, lr_b)
        loss     = l_pixel + 0.001 * l_vgg + 0.3 * l_struct

        opt_G2.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        opt_G2.step()
        losses.append(l_pixel.item())

    sch_G2.step()
    avg = sum(losses) / len(losses)
    print(f"P2 Ep {epoch:03d}/{PHASE2_EPOCHS} | L1={avg:.5f} | LR={sch_G2.get_last_lr()[0]:.6f}")

    if avg < best_loss_p2:
        best_loss_p2 = avg; save_G(G, "p2_best.pth")

    # MAE eval at ep25, ep50, ep75
    if epoch % 25 == 0 or epoch == PHASE2_EPOCHS:
        mae = full_eval_mae(G, TRAIN_LR_DIR, TRAIN_HR_DIR)
        if mae < best_mae_p2:
            best_mae_p2 = mae; save_G(G, "p2_best_mae.pth")
            print(f"  --> p2_best_mae.pth  (MAE={best_mae_p2:.4f})")

if not os.path.exists("p2_best_mae.pth"):
    src = "p2_best.pth" if os.path.exists("p2_best.pth") else "p1_best_mae.pth"
    shutil.copy(src, "p2_best_mae.pth")
    print(f"  [fallback] p2_best_mae.pth <- {src}")

print(f"\nPhase 2 done | best MAE={best_mae_p2:.4f}")

PHASE 2: Perceptual Refinement  (75 epochs, BS=64)
Loaded P1 weights: p1_best_mae.pth


P2 Ep 001/75 | L1=0.13707 | LR=0.000100


P2 Ep 002/75 | L1=0.13717 | LR=0.000100


P2 Ep 003/75 | L1=0.13721 | LR=0.000100


P2 Ep 004/75 | L1=0.13684 | LR=0.000099


...

P2 Ep 072/75 | L1=0.13589 | LR=0.000001


P2 Ep 073/75 | L1=0.13566 | LR=0.000001


P2 Ep 074/75 | L1=0.13580 | LR=0.000001


P2 Ep 075/75 | L1=0.13582 | LR=0.000001


  MAE: 17.3098
  [fallback] p2_best_mae.pth <- p2_best.pth

Phase 2 done | best MAE=17.1566


In [8]:
# ============================================================
# PHASE 3 — cGAN Fine-Tuning
# BS=32 → 52 batches/ep × ~2.1s/batch = ~109s/ep
# 55 ep × 109s = ~100 min
# ============================================================
print("=" * 65)
print(f"PHASE 3: cGAN Fine-Tuning  ({PHASE3_EPOCHS} epochs, BS={BS_P3})")
print("=" * 65)

for ckpt in ["p2_best_mae.pth", "p2_best.pth", "p1_best_mae.pth", "p1_best.pth"]:
    if os.path.exists(ckpt):
        load_G(G, ckpt); print(f"Loaded G weights: {ckpt}"); break

D      = wrap_dp(Discriminator().to(DEVICE))
D.apply(weight_init) if not hasattr(D, "module") else D.module.apply(weight_init)
loader = make_loader(BS_P3)

opt_G3 = optim.Adam(G.parameters(), lr=P3_LR_G, betas=(0.9, 0.999), weight_decay=1e-6)
opt_D  = optim.Adam(D.parameters(), lr=P3_LR_D, betas=(0.9, 0.999))
sch_G3 = optim.lr_scheduler.CosineAnnealingLR(opt_G3, T_max=PHASE3_EPOCHS, eta_min=1e-7)
sch_D  = optim.lr_scheduler.CosineAnnealingLR(opt_D,  T_max=PHASE3_EPOCHS, eta_min=1e-6)

best_mae    = best_mae_p2
best_loss   = float("inf")
p3_improved = False

for epoch in range(1, PHASE3_EPOCHS + 1):
    G.train(); D.train()
    g_losses, d_losses = [], []

    for lr_b, hr_b in tqdm(loader, desc=f"P3 Ep {epoch}/{PHASE3_EPOCHS}", leave=False):
        lr_b, hr_b = lr_b.to(DEVICE), hr_b.to(DEVICE)

        # ── D step ────────────────────────────────────────────────────────────
        with torch.no_grad():
            fake = G(lr_b)
        loss_D = lsgan_d_loss(D(lr_b, hr_b), D(lr_b, fake))
        opt_D.zero_grad(); loss_D.backward()
        torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
        opt_D.step()
        d_losses.append(loss_D.item())

        # ── G step ────────────────────────────────────────────────────────────
        pred     = G(lr_b)
        l_pixel  = F.l1_loss(pred, hr_b)
        l_vgg    = VGG_loss(pred, hr_b)
        l_struct = F.l1_loss(F.interpolate(pred, scale_factor=0.25,
                              mode="bicubic", align_corners=False), lr_b)
        l_adv    = lsgan_g_loss(D(lr_b, pred))
        loss_G   = l_pixel + 0.001 * l_vgg + 0.15 * l_struct + 0.002 * l_adv

        opt_G3.zero_grad(); loss_G.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        opt_G3.step()
        g_losses.append(l_pixel.item())

    sch_G3.step(); sch_D.step()
    avg_g = sum(g_losses) / len(g_losses)
    avg_d = sum(d_losses) / len(d_losses)
    print(f"P3 Ep {epoch:03d}/{PHASE3_EPOCHS} | G_L1={avg_g:.5f} | D={avg_d:.4f} | "
          f"LR_G={sch_G3.get_last_lr()[0]:.6f}")

    if avg_g < best_loss:
        best_loss = avg_g; save_G(G, "p3_best.pth")

    # MAE eval at ep15, ep30, ep45, ep55
    if epoch % 15 == 0 or epoch == PHASE3_EPOCHS:
        mae = full_eval_mae(G, TRAIN_LR_DIR, TRAIN_HR_DIR)
        if mae < best_mae:
            best_mae = mae; p3_improved = True
            save_G(G, "best_mae.pth")
            print(f"  --> best_mae.pth  (MAE={best_mae:.4f})")

if not p3_improved:
    print(f"\n[INFO] P3 did not improve over P2 ({best_mae:.4f}). Falling back to P2.")
    src = next((c for c in ["p2_best_mae.pth", "p2_best.pth"] if os.path.exists(c)), None)
    if src:
        shutil.copy(src, "best_mae.pth")
        print(f"  Copied {src} -> best_mae.pth")

print(f"\nPhase 3 done | global best MAE={best_mae:.4f}")

PHASE 3: cGAN Fine-Tuning  (55 epochs, BS=32)
Loaded G weights: p2_best_mae.pth


P3 Ep 001/55 | G_L1=0.13584 | D=0.5332 | LR_G=0.000050


P3 Ep 002/55 | G_L1=0.13601 | D=0.3280 | LR_G=0.000050


P3 Ep 003/55 | G_L1=0.13559 | D=0.2759 | LR_G=0.000050


P3 Ep 004/55 | G_L1=0.13575 | D=0.2465 | LR_G=0.000049


...

P3 Ep 052/55 | G_L1=0.13531 | D=0.0209 | LR_G=0.000000


P3 Ep 053/55 | G_L1=0.13558 | D=0.0223 | LR_G=0.000000


P3 Ep 054/55 | G_L1=0.13549 | D=0.0219 | LR_G=0.000000


P3 Ep 055/55 | G_L1=0.13543 | D=0.0219 | LR_G=0.000000


  MAE: 17.2672

[INFO] P3 did not improve over P2 (17.1566). Falling back to P2.
  Copied p2_best_mae.pth -> best_mae.pth

Phase 3 done | global best MAE=17.1566


In [9]:
# Load the very best model across all three phases
best_model = Generator(num_blocks=NUM_BLOCKS).to(DEVICE)
for ckpt in ["best_mae.pth", "p3_best.pth",
             "p2_best_mae.pth", "p2_best.pth",
             "p1_best_mae.pth", "p1_best.pth"]:
    if os.path.exists(ckpt):
        best_model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print(f"Loaded for submission: {ckpt}")
        break

best_model.eval()
print("\nFinal training-set MAE (no TTA):")
full_eval_mae(best_model, TRAIN_LR_DIR, TRAIN_HR_DIR)

Loaded for submission: best_mae.pth

Final training-set MAE (no TTA):


  MAE: 17.3100


17.309970968605974

In [10]:
@torch.no_grad()
def generate_submission(model, test_dir, out="submission.csv"):
    """7-fold TTA: original + hflip + vflip + hvflip + rot90/180/270."""
    model.eval()
    loader      = DataLoader(SRTestDataset(test_dir), batch_size=16,
                             shuffle=False, num_workers=2)
    ids, pixels = [], []

    for lr_imgs, fnames in tqdm(loader, desc="TTA (7-fold)"):
        lr_imgs = lr_imgs.to(DEVICE)
        p = [model(lr_imgs)]
        hf  = model(torch.flip(lr_imgs,[3]));       p.append(torch.flip(hf,[3]))
        vf  = model(torch.flip(lr_imgs,[2]));       p.append(torch.flip(vf,[2]))
        hvf = model(torch.flip(lr_imgs,[2,3]));     p.append(torch.flip(hvf,[2,3]))
        r90 = model(torch.rot90(lr_imgs,1,[2,3]));  p.append(torch.rot90(r90,-1,[2,3]))
        r180= model(torch.rot90(lr_imgs,2,[2,3]));  p.append(torch.rot90(r180,-2,[2,3]))
        r270= model(torch.rot90(lr_imgs,3,[2,3]));  p.append(torch.rot90(r270,-3,[2,3]))

        avg = (torch.stack(p).mean(0) + 1) / 2 * 255
        avg = avg.clamp(0, 255).round().cpu().numpy().astype(np.uint8)

        for i, fname in enumerate(fnames):
            img = avg[i].transpose(1, 2, 0)
            ids.append(fname)
            pixels.append(" ".join(map(str, img.flatten().tolist())))

    df = (pd.DataFrame({"Id": ids, "Pixels": pixels})
            .sort_values("Id").reset_index(drop=True))
    df.to_csv(out, index=False)
    print(f"Saved {len(df)} rows -> {out}")
    return df

sub = generate_submission(best_model, TEST_LR_DIR)
sub.head()

TTA (7-fold): 100%|██████████| 31/31 [00:11<00:00,  2.82it/s]


Saved 495 rows -> submission.csv


,Id,Pixels
0,agrivision_test_0000.png,157 162 155 156 159 154 154 156 152 151 153 15...
1,agrivision_test_0001.png,40 46 32 41 45 34 38 41 32 29 30 26 22 21 21 1...
2,agrivision_test_0002.png,111 126 100 114 128 103 117 131 107 121 135 11...
3,agrivision_test_0003.png,160 137 156 161 139 156 161 141 156 162 142 15...
4,agrivision_test_0004.png,129 106 114 128 105 113 126 105 112 125 105 11...
